# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [32]:
ПРІЗВИЩЕ_ІМЯ = 'Бісюк Роман'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-34'
НОМЕР        = 1       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [33]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 1   ·   Бісюк Роман   ·   ШІ-34

Ваші вісім стовпців:
   1. номер і/х
   2. Ускладнення
   3. ЛН
   4. Дата розтину
   5. spO2 без дот
   6. АТ пост
   7. АСАТ1
   8. АСАТ2

Ваша пара для аналізу зв’язку: Вік та spO2 без дот

Ваше додаткове питання:
   Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?


In [34]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,Ускладнення,ЛН,Дата розтину,spO2 без дот,АТ пост,АСАТ1,АСАТ2
0,2377/239,"тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС",ІІІ-IV,2022-03-11,90.0,140/100,36.2,24.1
1,1121/19,NaN,ІІ-ІІІ,NaT,83.0,130/80,44.0,66.4
2,283/674,NaN,ІІ,NaT,78.0,150/100,28.4,24.2
3,2372/513,NaN,ІІІ-IV,NaT,82.0,170/110,19.1,36.2
4,1441/671,NaN,ІІ-ІІІ,NaT,80.0,160/100,42.2,51.4
5,2573/354,"Респірат дестрес синдром, емфізема легень, СЛН",ІІ-ІІІ,2021-12-26,72.0,140/90,85.1,24.0
6,1762/555,NaN,ІІ-ІІІ,NaT,89.0,130/80,37.4,48.2
7,2664/348,NaN,І,NaT,90.0,140/80,36.2,15.3


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [35]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   номер і/х     121 non-null    object        
 1   Ускладнення   61 non-null     str           
 2   ЛН            121 non-null    str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   spO2 без дот  120 non-null    float64       
 5   АТ пост       120 non-null    str           
 6   АСАТ1         114 non-null    float64       
 7   АСАТ2         104 non-null    float64       
dtypes: datetime64[us](1), float64(3), object(1), str(3)
memory usage: 7.7+ KB


In [36]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- Ускладнення
    непорожніх 61 з 121 | унікальних 35 | типи: str×61
    приклади: ["'тромбоз підколінної артерії, гангрена? ТЕЛА? СЛН ГРДС'", "'Респірат дестрес синдром, емфізема легень, СЛН'", "'Респірат дестрес синдром, емфізема легень, СЛН ТІШ'", "'Гострий респіраторний дистрес синдром,ТЕЛА?Прогресуюча серц-леген недос'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(8

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [37]:
ПАСПОРТ = {
    'номер і/х':    ('ідентифікатор', 'Це просто номер запису пацієнта, рахувати з ним нічого не можна.'),
    'Ускладнення':  ('номінальна',    'Це назви ускладнень, їх можна лише порівняти: однакові чи ні.'),
    'ЛН':           ('порядкова',     'Ступені І, ІІ, ІІІ мають порядок, але відстань між ними невідома.'),
    'Дата розтину': ('інтервальна',   'Різниця дат має сенс (дні), але нуль календаря умовний.'),
    'spO2 без дот': ('абсолютна',     'Це відсоток від 0 до 100, нуль і 100 % природні.'),
    'АТ пост':      ('відношень',     'Тиск має справжній нуль, але записаний текстом «140/100».'),
    'АСАТ1':        ('відношень',     'Фермент має справжній нуль, 80 удвічі більше за 40.'),
    'АСАТ2':        ('відношень',     'Те саме, що АСАТ1, лише повторний аналіз.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,"Це просто номер запису пацієнта, рахувати з ним нічого н..."
1,Ускладнення,номінальна,"Це назви ускладнень, їх можна лише порівняти: однакові ч..."
2,ЛН,порядкова,"Ступені І, ІІ, ІІІ мають порядок, але відстань між ними ..."
3,Дата розтину,інтервальна,"Різниця дат має сенс (дні), але нуль календаря умовний."
4,spO2 без дот,абсолютна,"Це відсоток від 0 до 100, нуль і 100 % природні."
5,АТ пост,відношень,"Тиск має справжній нуль, але записаний текстом «140/100»."
6,АСАТ1,відношень,"Фермент має справжній нуль, 80 удвічі більше за 40."
7,АСАТ2,відношень,"Те саме, що АСАТ1, лише повторний аналіз."


In [38]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [39]:
# Порядок ступенів ЛН задаємо вручну (від легкого до важкого)
ПОРЯДОК_ЛН = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']

def підготувати(стовпець):
    """Прибирає порожні й виправляє дефекти запису."""
    з = df[стовпець].dropna()
    if стовпець == 'ЛН':            # ступінь -> номер у порядку (1..6)
        з = з.map({v: i for i, v in enumerate(ПОРЯДОК_ЛН, 1)})
    if стовпець == 'АТ пост':       # '140/100' -> 140 (систолічний)
        з = pd.to_numeric(з.str.split('/').str[0], errors='coerce').dropna()
    if стовпець == 'spO2 без дот':  # 0.9 -> 90 (записали часткою)
        з = з.where(з > 1, з * 100)
    return з


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = підготувати(стовпець)
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    # усі шкали: мода і частоти
    р['мода'] = з.mode()[0]
    р['частоти (топ-3)'] = з.value_counts().head(3).to_dict()

    # порядкова і вище: медіана, квартилі
    if ш in ('порядкова', 'інтервальна', 'відношень', 'абсолютна'):
        р['медіана'] = з.median()
        р['Q1'] = з.quantile(0.25)
        р['Q3'] = з.quantile(0.75)

    # інтервальна і вище: середнє, відхилення
    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        р['середнє'] = з.mean()
        р['ст. відхилення'] = з.std()

    # відношень і абсолютна: коефіцієнт варіації
    if ш in ('відношень', 'абсолютна'):
        р['коеф. варіації, %'] = round(з.std() / з.mean() * 100, 1)

    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- номер і/х  (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 119}

--- Ускладнення  (номінальна)
    {'непорожніх': 61, 'унікальних': 35, 'мода': 'Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок', 'частоти (топ-3)': {'Гострий респіраторний дистрес синдром. Гостра серц-леген недос .Гостра емфізема легень.Токсико-інфекційний шок': 14, 'ГРДС ': 5, 'Прогресуюча серц-леген недос .Набряк легень . Сладж синдром': 3}}

--- ЛН  (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': np.int64(4), 'частоти (топ-3)': {4: 59, 3: 21, 5: 17}, 'медіана': np.float64(4.0), 'Q1': np.float64(3.0), 'Q3': np.float64(4.0)}

--- Дата розтину  (інтервальна)
    {'непорожніх': 61, 'унікальних': 50, 'мода': Timestamp('2022-06-07 00:00:00'), 'частоти (топ-3)': {Timestamp('2022-06-07 00:00:00'): 3, Timestamp('2022-04-07 00:00:00'): 2, Timestamp('2022-04-27 00:00:00'): 2}, 'медіана': Timestamp('2022-04-23 00:00:00'), 'Q1': Timestamp('2022-0

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [40]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
номер і/х,121.0,119.0,1837.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Ускладнення,61,35,Гострий респіраторний дистрес синдром. Гостра серц-леген...,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,0.9,78.75,84.0,89.0,98.0,12.208249
АТ пост,120,23,130/80,33,NaN,NaN,NaN,NaN,NaN,NaN,NaN
АСАТ1,114.0,NaN,NaN,NaN,30.709912,0.1,16.15,30.2,42.2,123.0,24.029872
АСАТ2,104.0,NaN,NaN,NaN,32.983173,0.4,14.975,34.5,46.775,120.6,24.093824


In [41]:
БЕЗГЛУЗДІ = [
    ('номер і/х', 'середнє', pd.to_numeric(df['номер і/х'], errors='coerce').mean(),
     'Номер історії хвороби є лише міткою, тож «середній номер» нічого не означає.'),
    ('ЛН', 'середнє', підготувати('ЛН').mean(),
     'Ступені мають лише порядок, а ступеня «3.57» не існує.'),
    ('spO2 без дот', 'мінімум', df['spO2 без дот'].min(),
     'Значення 0.9 — це 90 %, записане часткою, а не реальна сатурація.'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,номер і/х,середнє,1770.123077,"Номер історії хвороби є лише міткою, тож «середній номер..."
1,ЛН,середнє,3.570248,"Ступені мають лише порядок, а ступеня «3.57» не існує."
2,spO2 без дот,мінімум,0.900000,"Значення 0.9 — це 90 %, записане часткою, а не реальна с..."


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [42]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Вік × spO2 без дот

Вік: шкала — не у вашому списку
spO2 без дот: шкала — абсолютна

   Вік  spO2 без дот
0   55          90.0
1   43          83.0
2   54          78.0
3   68          82.0
4   65          80.0
5   66          72.0


In [43]:
ОБРАНА_МІРА = 'кореляція Пірсона'
ЧОМУ_САМЕ_ВОНА = 'Вік має шкалу відношень, а spO2 абсолютну; обидві числові з природним нулем, тому Пірсон допустимий.'

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

from scipy import stats

пара = pd.DataFrame({'Вік': df['Вік'], 'spO2': підготувати('spO2 без дот')}).dropna()
r, p = stats.pearsonr(пара['Вік'], пара['spO2'])
print(f'Пірсон: r = {r:.2f}, p = {p:.4f}')
print('Що старший пацієнт, то нижча сатурація (зв’язок слабкий, але значущий).')

Пірсон: r = -0.27, p = 0.0027
Що старший пацієнт, то нижча сатурація (зв’язок слабкий, але значущий).


In [44]:
# Пірсон між номером історії хвороби та spO2 — недопустимо
номер = pd.to_numeric(df['номер і/х'], errors='coerce')
погано = pd.DataFrame({'номер': номер, 'spO2': підготувати('spO2 без дот')}).dropna()
r_погано, _ = stats.pearsonr(погано['номер'], погано['spO2'])
print(f'Пірсон номер і/х × spO2: r = {r_погано:.2f}')
print('Число пораховано, але номер є лише міткою: якщо перенумерувати пацієнтів, r зміниться. Тому інтерпретувати його не можна.')

Пірсон номер і/х × spO2: r = 0.13
Число пораховано, але номер є лише міткою: якщо перенумерувати пацієнтів, r зміниться. Тому інтерпретувати його не можна.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [45]:
ПІДГОТОВКА = {
    'номер і/х':    'Не беремо в модель, бо це ідентифікатор.',
    'Ускладнення':  'One-hot кодування, порожнє означає «немає ускладнень».',
    'ЛН':           'Цілі коди 1–6 за порядком ПОРЯДОК_ЛН.',
    'Дата розтину': 'Переводимо в кількість днів від поступлення.',
    'spO2 без дот': 'Виправляємо 0.9 на 90, потім нормалізуємо.',
    'АТ пост':      'Ділимо на систолічний і діастолічний, потім стандартизуємо.',
    'АСАТ1':        'Логарифмуємо (розкид великий), потім стандартизуємо.',
    'АСАТ2':        'Логарифмуємо (розкид великий), потім стандартизуємо.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,"Не беремо в модель, бо це ідентифікатор."
1,Ускладнення,номінальна,"One-hot кодування, порожнє означає «немає ускладнень»."
2,ЛН,порядкова,Цілі коди 1–6 за порядком ПОРЯДОК_ЛН.
3,Дата розтину,інтервальна,Переводимо в кількість днів від поступлення.
4,spO2 без дот,абсолютна,"Виправляємо 0.9 на 90, потім нормалізуємо."
5,АТ пост,відношень,"Ділимо на систолічний і діастолічний, потім стандартизуємо."
6,АСАТ1,відношень,"Логарифмуємо (розкид великий), потім стандартизуємо."
7,АСАТ2,відношень,"Логарифмуємо (розкид великий), потім стандартизуємо."


In [46]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал

# 1. ЛН (порядкова) -> коди 1..6 у заданому порядку
лн_код = df['ЛН'].map({v: i for i, v in enumerate(ПОРЯДОК_ЛН, 1)})

# 2. Дата розтину (інтервальна) -> кількість днів від поступлення
днів_до_розтину = (df['Дата розтину'] - df['Поступлення']).dt.days

# 3. spO2 без дот (абсолютна) -> виправляємо 0.9 і нормалізуємо до 0..1
spo2 = df['spO2 без дот'].where(df['spO2 без дот'] > 1, df['spO2 без дот'] * 100)
spo2_норм = (spo2 - spo2.min()) / (spo2.max() - spo2.min())

результат = pd.DataFrame({
    'ЛН': df['ЛН'], 'ЛН_код': лн_код,
    'Дата розтину': df['Дата розтину'], 'днів_до_розтину': днів_до_розтину,
    'spO2': df['spO2 без дот'], 'spO2_норм': spo2_норм.round(3),
})
результат.head(8)

,ЛН,ЛН_код,Дата розтину,днів_до_розтину,spO2,spO2_норм
0,ІІІ-IV,6,2022-03-11,15.0,90.0,0.833
1,ІІ-ІІІ,4,NaT,NaN,83.0,0.688
2,ІІ,3,NaT,NaN,78.0,0.583
3,ІІІ-IV,6,NaT,NaN,82.0,0.667
4,ІІ-ІІІ,4,NaT,NaN,80.0,0.625
5,ІІ-ІІІ,4,2021-12-26,11.0,72.0,0.458
6,ІІ-ІІІ,4,NaT,NaN,89.0,0.812
7,І,1,NaT,NaN,90.0,0.833


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [47]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [48]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [49]:
ВИПРАВЛЕННЯ = [
    (1, 'помер/жив — номінальна шкала, середнє недопустиме',
        '1.5 — це не «посередині між смертю й життям», а лише наслідок того, що померлих і живих майже порівну',
        'Частки: померло 61 з 121 (50.4 %), вижило 60 (49.6 %)'),
    (2, 'Одна стать записана різними текстами (ж/жін, ч/чол)',
        '«4 статі» — це дефект введення, насправді їх дві, і групи «жін»/«чол» не окремі',
        'Звести до двох категорій: жінок 75, чоловіків 46'),
    (3, 'ЛН/ступінь КТ — порядкова, але відсортовано за алфавітом і взято середнє',
        'Алфавіт ставить «важкий» перед «легким», є дублікати з пробілами й описками, а середнє кодів не має змісту',
        'Почистити назви, задати порядок легкий < середній < важкий < вкрай важкий < критичний і взяти медіану (середній)'),
    (4, 'КТ записано у двох кодуваннях: частки (0.4) і текст («30-40%»)',
        'to_numeric з coerce мовчки викинув 17 із 44 значень, тож 39.6 % пораховано лише на 27 пацієнтах',
        'Перетворити діапазони «30-40%» на середину (0.35) і лише потім рахувати'),
    (5, 'spO2 без дот містить 0.9 — це 90 %, записане часткою',
        'Одне хибне значення занижує середнє: 82.05 замість 82.79, а мінімум 0.9 % взагалі неможливий для живої людини',
        'Замінити 0.9 на 90 (або виключити) і звітувати медіану 84'),
    (6, 'Дата — інтервальна шкала, коефіцієнт варіації недопустимий',
        '0.595 % залежить лише від того, що відлік іде від 1970 року; з іншим нулем календаря число було б іншим',
        'Звітувати розмах дат (2021-02-25 … 2022-06-03) і std у днях (~113 днів)'),
    (7, 'Стать і помер/жив — номінальні, Пірсон недопустимий',
        '0.177 залежить від того, як закодували 0 і 1, і не є мірою зв’язку для категорій',
        'Таблиця спряженості, χ² = 3.09 (p = 0.079) та Крамер V = 0.16'),
    (8, 'Нормалізовано всі числові стовпці, зокрема сталий ФЕРИТ2 і номінальні коди',
        'У ФЕРИТ2 одне значення, max = min, тому ділення на нуль дає NaN, а «усі в [0, 1]» неправда',
        'Перевірити варіативність, сталі стовпці й номінальні коди (помер/жив) не нормалізувати'),
    (9, 'dropna() видаляє рядок за будь-який пропуск, а пропуски є в кожному рядку',
        'Лишилося 0 рядків, тобто «повністю заповнений набір» — порожня таблиця, придатна ні для чого',
        'Видаляти стовпці з великою кількістю пропусків або заповнювати їх, а не видаляти всі рядки'),
    (10, 'Дата розтину існує лише в померлих — це витік цільової змінної',
        'r = 1.00 означає лише «є розтин → помер», тобто це сама відповідь, а не предиктор',
        'Виключити Дата розтину з ознак; тоді найсильніший показник Лім2 (|r| = 0.645)'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"помер/жив — номінальна шкала, середнє недопустиме","1.5 — це не «посередині між смертю й життям», а лише нас...","Частки: померло 61 з 121 (50.4 %), вижило 60 (49.6 %)"
1,2,"Одна стать записана різними текстами (ж/жін, ч/чол)","«4 статі» — це дефект введення, насправді їх дві, і груп...","Звести до двох категорій: жінок 75, чоловіків 46"
2,3,"ЛН/ступінь КТ — порядкова, але відсортовано за алфавітом...","Алфавіт ставить «важкий» перед «легким», є дублікати з п...","Почистити назви, задати порядок легкий < середній < важк..."
3,4,КТ записано у двох кодуваннях: частки (0.4) і текст («30...,"to_numeric з coerce мовчки викинув 17 із 44 значень, тож...",Перетворити діапазони «30-40%» на середину (0.35) і лише...
4,5,"spO2 без дот містить 0.9 — це 90 %, записане часткою",Одне хибне значення занижує середнє: 82.05 замість 82.79...,Замінити 0.9 на 90 (або виключити) і звітувати медіану 84
5,6,"Дата — інтервальна шкала, коефіцієнт варіації недопустимий","0.595 % залежить лише від того, що відлік іде від 1970 р...",Звітувати розмах дат (2021-02-25 … 2022-06-03) і std у д...
6,7,"Стать і помер/жив — номінальні, Пірсон недопустимий","0.177 залежить від того, як закодували 0 і 1, і не є мір...","Таблиця спряженості, χ² = 3.09 (p = 0.079) та Крамер V =..."
7,8,"Нормалізовано всі числові стовпці, зокрема сталий ФЕРИТ2...","У ФЕРИТ2 одне значення, max = min, тому ділення на нуль ...","Перевірити варіативність, сталі стовпці й номінальні код..."
8,9,"dropna() видаляє рядок за будь-який пропуск, а пропуски ...","Лишилося 0 рядків, тобто «повністю заповнений набір» — п...",Видаляти стовпці з великою кількістю пропусків або запов...
9,10,Дата розтину існує лише в померлих — це витік цільової з...,"r = 1.00 означає лише «є розтин → помер», тобто це сама ...",Виключити Дата розтину з ознак; тоді найсильніший показн...


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [50]:
from scipy import stats

# Помилка 1: середнє номінальної -> частки
print('НЕПРАВИЛЬНО: середнє помер/жив =', round(df['помер/жив'].mean(), 2))
print('ПРАВИЛЬНО:')
print(df['помер/жив'].map({1: 'помер', 2: 'вижив'}).value_counts(normalize=True).round(3))
print()

# Помилка 7: Пірсон для двох номінальних -> χ² і Крамер
стать = df['Стать'].str.strip().str[0].map({'ж': 'жін', 'ч': 'чол'})
таблиця = pd.crosstab(стать, df['помер/жив'])
chi2, p, _, _ = stats.chi2_contingency(таблиця)
V = np.sqrt(chi2 / (таблиця.values.sum() * (min(таблиця.shape) - 1)))
print('НЕПРАВИЛЬНО: Пірсон = 0.177')
print(f'ПРАВИЛЬНО: χ² = {chi2:.2f}, p = {p:.3f}, Крамер V = {V:.2f}')
print()

# Помилка 10: прибираємо витік (Дата розтину)
дані = df.select_dtypes('number').drop(columns=['помер/жив'])
ціль = (df['помер/жив'] == 1).astype(int)
print('НЕПРАВИЛЬНО: найкращий предиктор — Дата розтину, r = 1.00')
print('ПРАВИЛЬНО (без Дати розтину):')
print(дані.corrwith(ціль).abs().sort_values(ascending=False).head(5).round(3))

НЕПРАВИЛЬНО: середнє помер/жив = 1.5
ПРАВИЛЬНО:
помер/жив
помер    0.504
вижив    0.496
Name: proportion, dtype: float64

НЕПРАВИЛЬНО: Пірсон = 0.177
ПРАВИЛЬНО: χ² = 3.09, p = 0.079, Крамер V = 0.16

НЕПРАВИЛЬНО: найкращий предиктор — Дата розтину, r = 1.00
ПРАВИЛЬНО (без Дати розтину):
Лім2     0.645
ІЛ6-2    0.616
Л2       0.480
Вік      0.449
П/Я2     0.395
dtype: float64


---
# Етап 7. Висновки та індивідуальне питання

In [51]:
ВИСНОВКИ = [
    'Тип даних у pandas не збігається зі шкалою: ЛН, Ускладнення й АТ пост мають однаковий тип str, хоча шкали в них різні (порядкова, номінальна та відношень), тому шкалу треба визначати за змістом, а не за dtype.',
    'Шкала обмежує допустимі статистики: для ЛН медіана 4 (ступінь ІІ-ІІІ) має сенс, а середнє 3.57 ні, бо відстані між ступенями невідомі; для номера і/х середнє 1770 безглузде взагалі, бо це лише мітка.',
    'Вік і spO2 мають числові шкали з природним нулем, тому для них допустимий Пірсон: r = −0.27 (p = 0.003), тобто що старший пацієнт, то нижча сатурація. Але цей результат з’явився лише після виправлення значення 0.9, а без нього зв’язок виглядав незначущим.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. Тип даних у pandas не збігається зі шкалою: ЛН, Ускладнення й АТ пост мають однаковий тип str, хоча шкали в них різні (порядкова, номінальна та відношень), тому шкалу треба визначати за змістом, а не за dtype.
2. Шкала обмежує допустимі статистики: для ЛН медіана 4 (ступінь ІІ-ІІІ) має сенс, а середнє 3.57 ні, бо відстані між ступенями невідомі; для номера і/х середнє 1770 безглузде взагалі, бо це лише мітка.
3. Вік і spO2 мають числові шкали з природним нулем, тому для них допустимий Пірсон: r = −0.27 (p = 0.003), тобто що старший пацієнт, то нижча сатурація. Але цей результат з’явився лише після виправлення значення 0.9, а без нього зв’язок виглядав незначущим.


In [52]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = (
    'До перетворення «Вік» має шкалу відношень: є природний нуль, і 66 років удвічі більше за 33. '
    'Після поділу на групи «до 40», «40–60», «понад 60» шкала стає порядковою: групи мають порядок, '
    'але «ширина» в них різна (до 40 охоплює ~40 років, 40–60 лише 20), тож відстані між ними невідомі. '
    'Втратили: точний вік (пацієнти 61 і 97 років тепер однакові), середнє, std та Пірсона; '
    'лишаються тільки медіана, частоти й Спірмен. До того ж групи вийшли нерівні: до 40 лише 3 пацієнти, '
    '40–60 — 37, понад 60 — 81. '
    'Здобули: групи легше пояснити лікарям, вони стійкі до помилок запису віку, '
    'і їх можна порівнювати з іншими категоріями (наприклад, помер/жив) через χ² і Крамера.'
)
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Стовпець «Вік» можна перетворити на вікові групи «до 40», «40–60», «понад 60». Яка шкала була до перетворення і яка стала? Що ви при цьому втратили і що здобули?

До перетворення «Вік» має шкалу відношень: є природний нуль, і 66 років удвічі більше за 33. Після поділу на групи «до 40», «40–60», «понад 60» шкала стає порядковою: групи мають порядок, але «ширина» в них різна (до 40 охоплює ~40 років, 40–60 лише 20), тож відстані між ними невідомі. Втратили: точний вік (пацієнти 61 і 97 років тепер однакові), середнє, std та Пірсона; лишаються тільки медіана, частоти й Спірмен. До того ж групи вийшли нерівні: до 40 лише 3 пацієнти, 40–60 — 37, понад 60 — 81. Здобули: групи легше пояснити лікарям, вони стійкі до помилок запису віку, і їх можна порівнювати з іншими категоріями (наприклад, помер/жив) через χ² і Крамера.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [53]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'текст, як є',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'код, перевірено',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'ні',
    'етап 7 — висновки й відповідь':   'текст, як є',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Запускав кожну клітинку окремо й порівнював результат із поясненнями. '
    'Знайшов помилку NameError в етапі 5 (бракувало рядків зі змінними) і виправив її. '
    'З’ясував, чому в Дата розтину так багато NaT (розтин лише в померлих), '
    'що означає spO2 без дотації та чому 0.9 є помилкою запису. '
    'Відхилив занадто прості висновки й попросив переписати їх так, щоб вони пов’язували шкали з результатами.'
)
ПІДТВЕРДЖУЮ = True      # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [54]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 1 готова до захисту.
Бісюк Роман, ШІ-34


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.